In [1]:
import pandas as pd

# Read each brand file
df_nykaa = pd.read_csv("../data/raw/nykaa.csv")
df_tira = pd.read_csv("../data/raw/tira.csv")
df_purplle = pd.read_csv("../data/raw/purplle.csv")

# Add a Brand column so we remember which file each row came from
df_nykaa["Brand"] = "Nykaa"
df_tira["Brand"] = "Tira"
df_purplle["Brand"] = "Purplle"

# Stack the three tables into one
df = pd.concat([df_nykaa, df_tira, df_purplle], ignore_index=True)
print(df.shape)

(166665, 17)


In [2]:
df.head()

,Campaign_ID,Campaign_Type,Target_Audience,Duration,Channel_Used,Impressions,Clicks,Leads,Conversions,Revenue,Acquisition_Cost,ROI,Language,Engagement_Score,Customer_Segment,Date,Brand
0,NY-CMP-1000,Social Media,College Students,21.0,"WhatsApp, YouTube",57804.0,6156.0,3616.0,2355.0,1867515.0,NaN,6.14,Hindi,20.98,College Students,29-04-2025,Nykaa
1,NY-CMP-1001,Paid Ads,Tier 2 City Customers,18.0,YouTube,91801.0,3321.0,1971.0,1357.0,1046247.0,180.83,3.26,Hindi,7.24,College Students,06-04-2025,Nykaa
2,NY-CMP-1002,Influencer,Youth,23.0,"WhatsApp, Google, YouTube",15536.0,2182.0,952.0,755.0,197055.0,90.60,NaN,English,25.03,College Students,14-01-2025,Nykaa
3,NY-CMP-1003,Email,Working Women,18.0,"YouTube, Facebook, Instagram",88114.0,8413.0,2231.0,947.0,376906.0,249.07,0.60,Hindi,13.15,NaN,04-06-2025,Nykaa
4,NY-CMP-1004,Paid Ads,College Students,10.0,"Facebook, Instagram",96871.0,3743.0,2060.0,1258.0,518296.0,228.60,0.80,Hindi,7.29,Tier 2 City Customers,29-12-2024,Nykaa


In [3]:
df.isnull().sum()

Campaign_ID         5417
Campaign_Type       8358
Target_Audience     8172
Duration            8323
Channel_Used        8282
Impressions         8224
Clicks              8224
Leads               8090
Conversions         8000
Revenue             8304
Acquisition_Cost    8161
ROI                 8298
Language            8299
Engagement_Score    8081
Customer_Segment    8175
Date                8099
Brand                  0
dtype: int64

In [4]:
# We clean the copy so the original df stays safe
df_clean = df.copy()

In [15]:
# Duplicate full rows
print("Duplicated rows:", df_clean.duplicated().sum())
# Duplicated Campaign_IDs (ignore the missing ones)
ids = df_clean["Campaign_ID"].dropna()
print("Duplicated IDs:",ids.duplicated().sum())


Duplicated rows: 0
Duplicated IDs: 0


In [16]:
# The ID is only a label, so "Unknown" is fine
df_clean["Campaign_ID"] = df_clean["Campaign_ID"].fillna("Unknown")

In [17]:
# The dates look like 29-04-2025 (day-month-year)
df_clean["Date"] = pd.to_datetime(df_clean["Date"], format="%d-%m-%Y")

df_clean["Year"] = df_clean["Date"].dt.year
df_clean["Month"] = df_clean["Date"].dt.month

# Missing dates give missing Year and Month, so fill with the most common value
df_clean["Year"] = df_clean["Year"].fillna(df_clean["Year"].mode()[0])
df_clean["Month"] = df_clean["Month"].fillna(df_clean["Month"].mode()[0])

In [18]:
# Total cost = cost per conversion x number of conversions
total_cost = df_clean["Acquisition_Cost"] * df_clean["Conversions"]

# ROI by formula
calculated_roi = (df_clean["Revenue"] - total_cost) / total_cost

# How far is the given ROI from the formula? (0.05 allows for rounding)
difference = (df_clean["ROI"] - calculated_roi).abs()
wrong_roi = difference > 0.05

print("Rows with wrong ROI:", wrong_roi.sum())

Rows with wrong ROI: 0


In [19]:
total_cost = df_clean["Acquisition_Cost"] * df_clean["Conversions"]

# Rows where ROI is missing but the other three values exist
can_fill = (
    df_clean["ROI"].isnull()
    & df_clean["Revenue"].notnull()
    & df_clean["Acquisition_Cost"].notnull()
    & df_clean["Conversions"].notnull()
)

new_roi = (df_clean["Revenue"] - total_cost) / total_cost

# .loc[rows, column] = values  -> fill only the chosen rows
df_clean.loc[can_fill, "ROI"] = new_roi[can_fill].round(2)

print("ROI still missing:", df_clean["ROI"].isnull().sum())

ROI still missing: 1234


In [20]:
total_cost = df_clean["Acquisition_Cost"] * df_clean["Conversions"]

can_fill = (
    df_clean["Revenue"].isnull()
    & df_clean["ROI"].notnull()
    & df_clean["Acquisition_Cost"].notnull()
    & df_clean["Conversions"].notnull()
)

# Revenue = total cost x (1 + ROI)
new_revenue = total_cost * (1 + df_clean["ROI"])
df_clean.loc[can_fill, "Revenue"] = new_revenue[can_fill].round(0)

print("Revenue still missing:", df_clean["Revenue"].isnull().sum())

Revenue still missing: 1167


In [22]:
can_fill = (
    df_clean["Acquisition_Cost"].isnull()
    & df_clean["Revenue"].notnull()
    & df_clean["ROI"].notnull()
    & df_clean["Conversions"].notnull()
)

# Cost = Revenue / (Conversions x (1 + ROI))
new_cost = df_clean["Revenue"] / (df_clean["Conversions"] * (1 + df_clean["ROI"]))
df_clean.loc[can_fill, "Acquisition_Cost"] = new_cost[can_fill].round(2)

print("Cost still missing:", df_clean["Acquisition_Cost"].isnull().sum())

Cost still missing: 1170


In [23]:
can_fill = (
    df_clean["Conversions"].isnull()
    & df_clean["Revenue"].notnull()
    & df_clean["ROI"].notnull()
    & df_clean["Acquisition_Cost"].notnull()
)

# Conversions = Revenue / (cost x (1 + ROI))
new_conversions = df_clean["Revenue"] / (df_clean["Acquisition_Cost"] * (1 + df_clean["ROI"]))
df_clean.loc[can_fill, "Conversions"] = new_conversions[can_fill].round(0)

print("Conversions still missing:", df_clean["Conversions"].isnull().sum())

Conversions still missing: 1154


In [24]:
df_clean["Duration"] = df_clean["Duration"].fillna(df_clean["Duration"].median())
df_clean["Impressions"] = df_clean["Impressions"].fillna(df_clean["Impressions"].median())
df_clean["Clicks"] = df_clean["Clicks"].fillna(df_clean["Clicks"].median())
df_clean["Leads"] = df_clean["Leads"].fillna(df_clean["Leads"].median())
df_clean["Conversions"] = df_clean["Conversions"].fillna(df_clean["Conversions"].median())
df_clean["Revenue"] = df_clean["Revenue"].fillna(df_clean["Revenue"].median())
df_clean["Acquisition_Cost"] = df_clean["Acquisition_Cost"].fillna(df_clean["Acquisition_Cost"].median())
df_clean["Engagement_Score"] = df_clean["Engagement_Score"].fillna(df_clean["Engagement_Score"].median())

In [25]:
total_cost = df_clean["Acquisition_Cost"] * df_clean["Conversions"]
new_roi = (df_clean["Revenue"] - total_cost) / total_cost

roi_missing = df_clean["ROI"].isnull()
print("ROI rows to calculate:", roi_missing.sum())

df_clean.loc[roi_missing, "ROI"] = new_roi[roi_missing].round(2)

ROI rows to calculate: 1234


In [26]:
df_clean["Campaign_Type"] = df_clean["Campaign_Type"].fillna("Unknown")
df_clean["Target_Audience"] = df_clean["Target_Audience"].fillna("Unknown")
df_clean["Channel_Used"] = df_clean["Channel_Used"].fillna("Unknown")
df_clean["Language"] = df_clean["Language"].fillna("Unknown")
df_clean["Customer_Segment"] = df_clean["Customer_Segment"].fillna("Unknown")

In [27]:
# These are counts, so they should be whole numbers
df_clean["Duration"] = df_clean["Duration"].astype(int)
df_clean["Impressions"] = df_clean["Impressions"].astype(int)
df_clean["Clicks"] = df_clean["Clicks"].astype(int)
df_clean["Leads"] = df_clean["Leads"].astype(int)
df_clean["Conversions"] = df_clean["Conversions"].astype(int)
df_clean["Year"] = df_clean["Year"].astype(int)
df_clean["Month"] = df_clean["Month"].astype(int)

In [28]:
print(df_clean.shape)
print(df_clean.isnull().sum())

(166665, 19)
Campaign_ID            0
Campaign_Type          0
Target_Audience        0
Duration               0
Channel_Used           0
Impressions            0
Clicks                 0
Leads                  0
Conversions            0
Revenue                0
Acquisition_Cost       0
ROI                    0
Language               0
Engagement_Score       0
Customer_Segment       0
Date                8099
Brand                  0
Year                   0
Month                  0
dtype: int64


In [29]:
df_clean.to_csv("../data/processed/campaigns_clean.csv", index=False)
print("Saved!")

Saved!
